In [67]:
import pandas as pd
df = pd.read_csv("/Volumes/mac1/data/limb/best_hpo_terms_with_parents.csv")
#df.head()

In [68]:
import ast

df["Parents_list"] = df["Parents"].apply(ast.literal_eval)

def extract_cluster(parents):
    parents = [p.lower() for p in parents]
    if any(x in p for p in parents for x in [
            "glucose", "insulin", "hyperglycemia", "diabetes", "metabolism/homeostasis"
        ]):
            return "Glucose/Insulin"
    
    if any(x in p for p in parents for x in [
        "neoplasm", "tumor", "cancer", "malignancy", "carcinoma", "sarcoma", "adenoma", "lymphoma", "melanoma"
    ]):
        return "Neoplasm"

    if any(x in p for p in parents for x in ["renal", "urinary", "kidney", "ureter"]):
        return "Renal"

    if any(x in p for p in parents for x in ["cardiac", "heart", "aorta", "ventricle", "septum", "artery", "vascular"]):
        return "Cardiac"

    if any(x in p for p in parents for x in ["skeletal", "bone", "vertebra", "sternum", "clavicle", 'growth', 'joint',"springel","pes","tarsa","carpa"
                                            "limb", "digit", "phalange", "metacarpal", "metatarsal", "toe", "finger","joint","sacro","valgus","varus"]):
        return "Skeletal"

    if any(x in p for p in parents for x in ["ear", "cochlea", "auditory"]):
        return "Auditory"

    if any(x in p for p in parents for x in ["eye", "iris", "optic", "lens"]):
        return "Ocular"

    if any(x in p for p in parents for x in ["genitalia", "uterus", "vagina", "scrotum", "reproductive"]):
        return "Genital"

    if any(x in p for p in parents for x in ["pancreas", "intestine", "colon", "abdominal", "digestive"]):
        return "GI"

    # Blood-related
    if any(x in p for p in parents for x in [
        "hemato", 'anemia', 'leukemia', 'lymphoma', 'thrombocytopenia', 'coagulation'
    ]):
        return "Blood"

    # Endocrine
    if any(x in p for p in parents for x in [
        "endocrine", "pituitary", "thyroid", "parathyroid", "gland",'puberty', 'hormone', 'adrenal', 'cortisol', 'aldosterone', 'androgen', 'estrogen', 'testosterone'
    ]):
        return "Endocrine"

    # Skin / dermatologic
    if any(x in p for p in parents for x in [
        "skin", "derm", "integument", "nail", "adnexa", 'hair'
    ]):
        return "Skin"

    # Placenta / umbilical cord / pregnancy
    if any(x in p for p in parents for x in [
        "placenta", "umbilical", "prenatal", "fetal", "pregnancy", "gestation"
    ]):
        return "Placenta/Prenatal"

    if any(x in p for p in parents for x in [
            "palate", "pharynx", "larynx", "esophagus", "throat", 'pulmon', 'sinus', 'trachea', 'broncho', 'alveol', 'lung', 'respiratory', 'nasal' 
        ]):
            return "Oral/Respiratory"
    if any(x in p for p in parents for x in [
            "face", "nose", "mouth", "langue", 'toungue', 'cheek', 'jaw', 'mandible', 'maxilla', 'lip', 'palate', 'philturm', 'pit', 'auricular'
        ]):
            return "Face"
    if any(x in p for p in parents for x in [
                "brain", "cerebrum", "cerebellum", "spinal cord", "nervous system", "myelo", "neuron", "nerve","autism"
            ]):
                return "Nervous System"
    return "Other"



df["Cluster"] = df["Parents_list"].apply(extract_cluster)


In [69]:
df.to_csv("/Volumes/mac1/data/limb/best_hpo_terms_with_parents_and_organs.csv", index=False)

In [59]:
import plotly.express as px

fig = px.sunburst(
    df,
    path=["Cluster", "Term"],
    values="Combined Score",
    color="Cluster"
)
import plotly.io as pio
pio.renderers.default = "browser"

fig.show()


In [70]:
import re
df["TF_family"] = df["Genes"].apply(lambda x: [re.sub(r"[\d-]+$", "", tf) for tf in x.split(";")])

In [72]:
df.to_csv("/Volumes/mac1/data/limb/best_hpo_terms_with_parents_and_clusters_and_TFfamilies.csv")


In [73]:
df= pd.read_csv("/Volumes/mac1/data/limb/best_hpo_terms_with_parents_and_clusters_and_TFfamilies.csv")
#df.head()

In [62]:
import plotly.express as px

fig = px.sunburst(
    df,
    path=["Cluster", "Term", "TF_family"],
    values="Combined Score",
    color="Cluster"
)

fig.update_layout(
    margin=dict(t=0, l=0, r=0, b=0)
)

fig.show()


In [63]:
df.to_csv("/Volumes/mac1/data/limb/best_hpo_terms_with_parents_and_clusters_and_TFfamilies.csv", index=False)

In [64]:
import pandas as pd
df= pd.read_csv("/Volumes/mac1/data/limb/best_hpo_terms_with_parents_and_clusters_and_TFfamilies.csv")

In [65]:
df.Term[1].split("(")[0].strip()

'Abnormal pancreas size'

In [75]:
from pyvis.network import Network
import networkx as nx
import ast
import numpy as np

# --- Build graph with top 150 HPO ---
df_top = df.nlargest(30, "Combined Score")
df_top = df_top[df_top["Cluster"] != "Other"]
#df_top = df

G = nx.Graph()

# --- Add TF nodes ---
for tf_list in df_top["TF_family"]:
    tf_list = ast.literal_eval(tf_list)
    for tf in tf_list:
        G.add_node(tf, type="TF")

# --- Add Cluster nodes ---
for cl in df_top["Cluster"].unique():
    G.add_node(cl, type="Cluster")

# --- Add HPO nodes ---
for term in df_top["Term"].unique():
    G.add_node(term, type="HPO")

# --- Add weighted edges ---
for _, row in df_top.iterrows():
    tf_list = ast.literal_eval(row["TF_family"])
    for tf in tf_list:
        G.add_edge(tf, row["Cluster"], combined=row["Combined Score"])
    G.add_edge(row["Cluster"], row["Term"], combined=row["Combined Score"])

# --- 2D layout ---
pos2d = nx.kamada_kawai_layout(G)

# --- Add pseudo‑3D depth ---
pos3d = {n: np.append(pos2d[n], np.random.uniform(-1, 1)) for n in G.nodes}

# --- PyVis graph ---
net = Network(
    height="900px",
    width="100%",
    bgcolor="#ffffff",
    font_color="black",
    cdn_resources="in_line"
)

# --- Activate physics (required for movement) ---
net.set_options("""
{
"nodes": {
    "font": {"size": 25}
  },
  "physics": {
    "enabled": true,
    "barnesHut": {
      "gravitationalConstant": -8000,
      "centralGravity": 0.3,
      "springLength": 95,
      "springConstant": 0.04,
      "damping": 0.09
    }
  }
}
""")

# --- Add nodes with pseudo‑3D coordinates ---
for n in G.nodes:
    x, y, z = pos3d[n]
    net.add_node(
        n,
        x=float(x),
        y=float(y),
        z=float(z),
        color=("red" if G.nodes[n]["type"]=="TF"
               else "blue" if G.nodes[n]["type"]=="Cluster"
               else "green"),
        size=(25 if G.nodes[n]["type"]=="TF"
              else 20 if G.nodes[n]["type"]=="Cluster"
              else 10)
    )

# --- Add edges with weights ---
for u, v, data in G.edges(data=True):
    score = data["combined"]
    net.add_edge(
        u, v,
        value=score,
        width=score/20,
        title=f"Combined Score: {score:.2f}",
        color="gray"
    )

net.write_html("/Volumes/mac1/data/limb/network_pseudo3D.html")


In [6]:
rows = [
    {"source": u, "target": v, **attrs}
    for u, v, attrs in G.edges(data=True)
]

df = pd.DataFrame(rows)
df.to_csv("/Volumes/mac1/data/limb/results/edges.csv", index=False)


« Le graphe pondéré TF–Cluster–HPO montre que les relations ne sont pas uniformes :
les arêtes les plus épaisses correspondent aux phénotypes les plus enrichis.
Les TFs TBX, GATA, HOXD et HNF1B apparaissent comme des hubs majeurs,
connectant plusieurs clusters avec des poids élevés.
Cela révèle des modules transcriptionnels cohérents et des axes biologiques partagés. »

In [88]:
df.head()

,Gene_set,Term,Overlap,P-value,Adjusted P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,Parents,Parents_list,Cluster,TF_family
0,Human_Phenotype_Ontology,Morphological abnormality of the inner ear (HP...,3/11,0.000306,0.034343,0,0,29.608500,239.572313,FOXI1;SOX10;POU3F4,"['Abnormal inner ear morphology', 'Abnormality...","['Abnormal inner ear morphology', 'Abnormality...",Auditory,"['FOXI', 'SOX', 'POU3F']"
1,Human_Phenotype_Ontology,Abnormal pancreas size (HP:0012094),3/11,0.000306,0.034343,0,0,29.608500,239.572313,PDX1;GATA6;HNF1B,"['Abnormal pancreas size', 'Abnormal pancreas ...","['Abnormal pancreas size', 'Abnormal pancreas ...",GI,"['PDX', 'GATA', 'HNF1B']"
2,Human_Phenotype_Ontology,Ureteral stenosis (HP:0000071),3/14,0.000657,0.049109,0,0,21.530182,157.778151,HOXD13;HNF1B;HOXA13,"['Ureteral stenosis', 'Ureteral obstruction', ...","['Ureteral stenosis', 'Ureteral obstruction', ...",Renal,"['HOXD', 'HNF1B', 'HOXA']"
3,Human_Phenotype_Ontology,Bifid scrotum (HP:0000048),4/22,0.000153,0.034343,0,0,17.607318,154.696128,HOXD13;SOX9;HOXA13;MNX1,"['Bifid scrotum', 'Abnormal scrotum morphology...","['Bifid scrotum', 'Abnormal scrotum morphology...",Renal,"['HOXD', 'SOX', 'HOXA', 'MNX']"
4,Human_Phenotype_Ontology,Concave nail (HP:0001598),2/8,0.004244,0.119014,0,0,26.216467,143.199698,LMX1B;HOXC13,"['Concave nail', 'Nail dysplasia', 'Abnormal n...","['Concave nail', 'Nail dysplasia', 'Abnormal n...",Skin,"['LMX1B', 'HOXC']"


In [94]:
import ast

def tf_union_from_terms_and_clusters(df, term_keywords, clusters):
    tf_set = set()

    # 1) Terms contenant les mots-clés (vertebr, trache, esoph, anal)
    mask_terms = df["Term"].str.contains("|".join(term_keywords), case=False, na=False)
    print(f"Nombre de termes correspondant aux mots-clés {term_keywords}: {df[mask_terms]}")
    sub_terms = df[mask_terms]

    for tf_str in sub_terms["TF_family"]:
        for tf in ast.literal_eval(tf_str):
            if tf != "Other":
                tf_set.add(tf)

    # 2) Clusters Cardiac + Skeletal
    mask_clusters = df["Cluster"].isin(clusters)
    sub_clusters = df[mask_clusters]

    for tf_str in sub_clusters["TF_family"]:
        for tf in ast.literal_eval(tf_str):
            if tf != "Other":
                tf_set.add(tf)

    return tf_set


# --- VACTERL definition précise ---
vacterl_term_keywords = ["vertebr", "trache", "esoph", "anal"]
vacterl_clusters = ["Cardiac", "Skeletal"]

tf_vacterl = tf_union_from_terms_and_clusters(
    df,
    term_keywords=vacterl_term_keywords,
    clusters=vacterl_clusters
)

print("TFs actifs dans VACTERL (définition précise) :")
print(sorted(tf_vacterl))


Nombre de termes correspondant aux mots-clés ['vertebr', 'trache', 'esoph', 'anal']:                      Gene_set  \
27   Human_Phenotype_Ontology   
49   Human_Phenotype_Ontology   
54   Human_Phenotype_Ontology   
57   Human_Phenotype_Ontology   
122  Human_Phenotype_Ontology   
148  Human_Phenotype_Ontology   
174  Human_Phenotype_Ontology   
234  Human_Phenotype_Ontology   
235  Human_Phenotype_Ontology   
272  Human_Phenotype_Ontology   
286  Human_Phenotype_Ontology   
295  Human_Phenotype_Ontology   
310  Human_Phenotype_Ontology   
350  Human_Phenotype_Ontology   
437  Human_Phenotype_Ontology   

                                                  Term Overlap   P-value  \
27                      Tracheal stenosis (HP:0002777)    3/23  0.002938   
49        Abnormal vertebral ossification (HP:0100569)    2/17  0.019128   
54   Abnormality of the intervertebral disk (HP:000...    2/19  0.023656   
57   Aplasia/Hypoplasia involving the vertebral col...    3/36  0.010499   
122   

In [116]:
fanconi_term_keywords = [
     "radial", "radius",
    "anemia", "erythro", "hemato", "blood",
    "tumor", "neoplasm", "malign"
]



tf_fanconi = tf_union_from_terms_and_clusters(
    df,
    term_keywords=fanconi_term_keywords,
    clusters=[]
)

print("TFs actifs dans Fanconi :", sorted(tf_fanconi))


Nombre de termes correspondant aux mots-clés ['radial', 'radius', 'anemia', 'erythro', 'hemato', 'blood', 'tumor', 'neoplasm', 'malign']:                      Gene_set  \
14   Human_Phenotype_Ontology   
84   Human_Phenotype_Ontology   
101  Human_Phenotype_Ontology   
179  Human_Phenotype_Ontology   
213  Human_Phenotype_Ontology   
257  Human_Phenotype_Ontology   
270  Human_Phenotype_Ontology   
312  Human_Phenotype_Ontology   
340  Human_Phenotype_Ontology   
382  Human_Phenotype_Ontology   
383  Human_Phenotype_Ontology   
386  Human_Phenotype_Ontology   
396  Human_Phenotype_Ontology   
418  Human_Phenotype_Ontology   
423  Human_Phenotype_Ontology   
425  Human_Phenotype_Ontology   

                                                  Term Overlap   P-value  \
14   Abnormal umbilical cord blood vessels (HP:0011...    2/10  0.006708   
84                           Hematochezia (HP:0002573)     1/8  0.096847   
101                     Microcytic anemia (HP:0001935)    2/25  0.039502

In [101]:
tar_term_keywords = [
    "radial", "radius",
    "anemia", "thrombocytopenia",
    "hypospadias"
]

tar_clusters = [
    "Renal",
    "Blood"
]

tf_tar = tf_union_from_terms_and_clusters(
    df,
    term_keywords=tar_term_keywords,
    clusters=[]
)

print("TFs actifs dans TAR :", sorted(tf_tar))


Nombre de termes correspondant aux mots-clés ['radial', 'radius', 'anemia', 'thrombocytopenia', 'hypospadias']:                      Gene_set                                         Term  \
101  Human_Phenotype_Ontology               Microcytic anemia (HP:0001935)   
213  Human_Phenotype_Ontology      Radial deviation of finger (HP:0009466)   
312  Human_Phenotype_Ontology              Hypochromic anemia (HP:0001931)   
355  Human_Phenotype_Ontology                     Hypospadias (HP:0000047)   
382  Human_Phenotype_Ontology  Abnormality of the radial head (HP:0003995)   
386  Human_Phenotype_Ontology                   Absent radius (HP:0003974)   

    Overlap   P-value  Adjusted P-value  Old P-value  Old Adjusted P-value  \
101    2/25  0.039502          0.237368            0                     0   
213   8/229  0.008920          0.141338            0                     0   
312    1/16  0.184348          0.356512            0                     0   
355    3/84  0.090609        

In [117]:
common_all = set(tf_fanconi) & set(tf_vacterl) & set(tf_tar)
common_all


{'GATA', 'HMGA', 'HOXA', 'HOXD', 'IRX', 'LMX1B', 'SMAD', 'TBX'}

In [118]:
#🧬 5. TFs communs VACTERL ↔ TAR (mais pas Fanconi)
common_vacterl_tar = set(tf_vacterl) & set(tf_tar) - set(tf_fanconi)
common_vacterl_tar

{'HNF1B'}

In [119]:
#specific to Fanconi (not in VACTERL or TAR)
set(tf_fanconi) - set(tf_vacterl) - set(tf_tar)


{'HNF1A', 'PHOX2B', 'POU6F'}

In [120]:
import networkx as nx

G = nx.Graph()

# Ajouter les trois syndromes
syndromes = ["Fanconi", "VACTERL", "TAR"]
for s in syndromes:
    G.add_node(s, type="syndrome", color="blue", size=40)

# Ajouter les TF et leurs liens
def add_tf_links(tf_set, syndrome_name):
    for tf in tf_set:
        if tf not in G.nodes:
            G.add_node(tf, type="TF", color="red", size=20)
        G.add_edge(tf, syndrome_name)

add_tf_links(tf_fanconi, "Fanconi")
add_tf_links(tf_vacterl, "VACTERL")
add_tf_links(tf_tar, "TAR")


In [121]:
from pyvis.network import Network

net = Network(
    height="900px",
    width="100%",
    bgcolor="#ffffff",
    font_color="black",
    cdn_resources="in_line"
)

# Options de physique
net.set_options("""
{
  "physics": {
    "enabled": true,
    "barnesHut": {
      "gravitationalConstant": -8000,
      "centralGravity": 0.3,
      "springLength": 95,
      "springConstant": 0.04,
      "damping": 0.09
    }
  }
}
""")

# Ajouter les nœuds
for n, data in G.nodes(data=True):
    net.add_node(
        n,
        color=data["color"],
        size=data["size"],
        title=n
    )

# Ajouter les arêtes
for u, v in G.edges():
    net.add_edge(u, v, color="gray")

net.write_html("network_tf_syndrome.html")


In [122]:
import networkx as nx
from pyvis.network import Network

G = nx.Graph()

# Ajouter les syndromes
syndromes = ["Fanconi", "VACTERL", "TAR"]
for s in syndromes:
    G.add_node(s, type="syndrome", color="blue", size=40)

# Fonction pour ajouter les TF
def add_tf_links(tf_set, syndrome_name):
    for tf in tf_set:
        if tf not in G.nodes:
            G.add_node(tf, type="TF", color="red", size=20)
        G.add_edge(tf, syndrome_name)

add_tf_links(tf_fanconi, "Fanconi")
add_tf_links(tf_vacterl, "VACTERL")
add_tf_links(tf_tar, "TAR")

# PyVis
net = Network(
    height="900px",
    width="100%",
    bgcolor="#ffffff",
    font_color="black",
    cdn_resources="in_line"
)

net.set_options("""
{
  "physics": {
    "enabled": true,
    "barnesHut": {
      "gravitationalConstant": -8000,
      "centralGravity": 0.3,
      "springLength": 95,
      "springConstant": 0.04,
      "damping": 0.09
    }
  }
}
""")

# Ajouter les nœuds
for n, data in G.nodes(data=True):
    net.add_node(
        n,
        color=data["color"],
        size=data["size"],
        title=n
    )

# Ajouter les arêtes
for u, v in G.edges():
    net.add_edge(u, v, color="gray")

net.write_html("network_tf_syndrome_final.html")


In [123]:
organ_systems = {
    "Skeletal": ["skeletal", "bone", "vertebr", "limb", "radial"],
    "Cardiac": ["cardiac", "heart"],
    "Renal": ["renal", "kidney", "ureter", "hypospadias"],
    "Blood": ["anemia", "hemato", "thrombocytopenia"],
    "GI": ["esoph", "trache", "anal", "digestive"],
    "Respiratory": ["respir", "lung", "trache"]
}
import ast

def tfs_for_organ(df, keywords):
    mask = df["Term"].str.contains("|".join(keywords), case=False, na=False)
    sub = df[mask]
    tf_set = set()
    for tf_str in sub["TF_family"]:
        for tf in ast.literal_eval(tf_str):
            if tf != "Other":
                tf_set.add(tf)
    return tf_set
import pandas as pd

# Collect all TFs from Fanconi, VACTERL, TAR
all_tfs = sorted(tf_fanconi | tf_vacterl | tf_tar)

# Build matrix
rows = []
for tf in all_tfs:
    row = {"TF": tf}
    for organ, keywords in organ_systems.items():
        row[organ] = int(tf in tfs_for_organ(df, keywords))
    rows.append(row)

df_heat = pd.DataFrame(rows).set_index("TF")
import plotly.express as px

fig = px.imshow(
    df_heat.T,
    labels=dict(x="TF", y="Organ System", color="Presence"),
    x=df_heat.index,
    y=df_heat.columns,
    color_continuous_scale=["white","red"]
)

fig.update_layout(
    title="Heatmap TF × Organ System",
    width=1400,
    height=700
)

fig.show()


In [3]:
import pandas as pd
df = pd.read_csv("/Volumes/mac1/data/limb/best_hpo_terms_with_parents_and_clusters_and_TFfamilies.csv")

In [4]:
import ast

# TF présents dans le cluster Diabetes/Glucose
diabetes_tfs = set()

for _, row in df[df["Cluster"]=="Glucose/Insulin"].iterrows():
    diabetes_tfs.update(ast.literal_eval(row["TF_family"]))

# HPO skeletiques qui partagent au moins un TF avec Diabetes
shared_hpo = []

for _, row in df[df["Cluster"]=="Skeletal"].iterrows():

    tfs = set(ast.literal_eval(row["TF_family"]))

    common = tfs & diabetes_tfs

    if len(common) > 0:
        shared_hpo.append({
            "Term": row["Term"],
            "Shared_TFs": ", ".join(sorted(common)),
            "N_shared": len(common)
        })

shared_df = pd.DataFrame(shared_hpo)
shared_df.sort_values("N_shared", ascending=False)

,Term,Shared_TFs,N_shared
23,Radial deviation of finger (HP:0009466),"GATA, LMX1B, SMAD, TBX",4
25,Intrauterine growth retardation (HP:0001511),"GATA, PDX, SMAD, TBX",4
24,Clinodactyly of the 5th finger (HP:0004209),"GATA, LMX1B, TBX",3
41,Short neck (HP:0000470),"HNF1B, SMAD, TBX",3
28,Mandibular prognathia (HP:0000303),"FOXG, HNF1B, SMAD",3
46,Brachydactyly syndrome (HP:0001156),"FOXG, GATA, SMAD",3
20,Umbilical hernia (HP:0001537),"GATA, TBX",2
8,Patellar aplasia (HP:0006443),"LMX1B, TBX",2
11,Aplasia/Hypoplasia of the patella (HP:0006498),"LMX1B, TBX",2
12,Patellar dislocation (HP:0002999),"LMX1B, TBX",2


In [8]:
shared_df.Term.to_list()

['Absent phalangeal crease (HP:0006109)',
 'Sprengel anomaly (HP:0000912)',
 'Long toe (HP:0010511)',
 'Abnormal finger flexion creases (HP:0006143)',
 'Hypoplastic iliac wings (HP:0002866)',
 'Aplasia of the musculature (HP:0100854)',
 'Gout (HP:0001997)',
 'Camptodactyly of toe (HP:0001836)',
 'Patellar aplasia (HP:0006443)',
 'Vertebral arch anomaly (HP:0008438)',
 'Overlapping toe (HP:0001845)',
 'Aplasia/Hypoplasia of the patella (HP:0006498)',
 'Patellar dislocation (HP:0002999)',
 'Joint contractures involving the joints of the feet (HP:0100492)',
 'Flexion contracture of toe (HP:0005830)',
 'Vertebral fusion (HP:0002948)',
 'Tetany (HP:0001281)',
 'Reduced subcutaneous adipose tissue (HP:0003758)',
 'Congenital diaphragmatic hernia (HP:0000776)',
 'Hypoplastic ilia (HP:0000946)',
 'Umbilical hernia (HP:0001537)',
 'Malar flattening (HP:0000272)',
 'Abnormality of the wing of the ilium (HP:0011867)',
 'Radial deviation of finger (HP:0009466)',
 'Clinodactyly of the 5th finger (H

In [10]:
from collections import Counter

counter = Counter()

for _, row in df[df["Cluster"]=="Skeletal"].iterrows():
    common = set(ast.literal_eval(row["TF_family"])) & diabetes_tfs

    for tf in common:
        counter[tf] += 1

counter.most_common()


[('LMX1B', 17),
 ('TBX', 17),
 ('SMAD', 15),
 ('GATA', 7),
 ('FOXG', 6),
 ('HNF1B', 4),
 ('PPARG', 2),
 ('PDX', 1)]